In [10]:
import sqlite3
import pandas as pd

# ==========================================
# 1. CONNECT TO THE DATABASE
# ==========================================
db_path = "chembl_36/chembl_36_sqlite/chembl_36.db"
print(f"Connecting to local database at: {db_path}")

conn = sqlite3.connect(db_path)
print("Successfully connected.")

# ==========================================
# 2. METAL-FOCUSED SQL QUERY
# ==========================================
# Common therapeutic transition/heavy metals used in organometallic chemistry.
# We scan ChEMBL's 'full_molformula' column directly inside SQL.
metal_symbols = ["Pt", "Ru", "Ir", "Au", "Pd", "Os", "Rh", "Re", "Fe", "Co", "Ni"]
metal_conditions = " OR ".join([f"prop.full_molformula LIKE '%{m}%'" for m in metal_symbols])

query = f"""    
SELECT DISTINCT 
    m.chembl_id, 
    str.canonical_smiles, 
    prop.full_molformula AS chemical_formula,
    act.standard_value AS ic50_nM, 
    act.pchembl_value AS pIC50,
    a.description AS assay_description,
    t.pref_name AS target_name
FROM activities act
JOIN molecule_dictionary m ON act.molregno = m.molregno
JOIN assays a ON act.assay_id = a.assay_id
JOIN target_dictionary t ON a.tid = t.tid
LEFT JOIN compound_structures str ON m.molregno = str.molregno
LEFT JOIN compound_properties prop ON m.molregno = prop.molregno
WHERE act.standard_type = 'IC50'
  AND act.standard_units = 'nM'
  AND t.organism = 'Homo sapiens'
  AND act.standard_value > 0
  AND ({metal_conditions});
"""

print("Executing targeted SQL extraction for metal-bound compounds...")
df = pd.read_sql_query(query, conn)
conn.close()
print(f"Pulled {len(df)} initial human records containing target metals in their formulas.")

# ==========================================
# 3. POST-PROCESSING AND DATA CLEANING
# ==========================================
if len(df) == 0:
    print("\nNo records matched the SQL query. Verify that your target metal symbols exist in the data.")
else:
    # 3a. Cancer Keyword Filter
    cancer_keywords = [
        "cancer", "tumor", "carcinoma", "cell line", "hela", "mcf", 
        "a549", "hct", "pc3", "cytotoxic", "antiproliferative", 
        "sarcoma", "melanoma", "leukemia"
    ]

    def is_cancer_related(text):
        if not isinstance(text, str):
            return False
        text_lower = text.lower()
        return any(kw in text_lower for kw in cancer_keywords)

    print("Filtering for cancer-related assays...")
    df["is_cancer"] = df["assay_description"].apply(is_cancer_related)
    df_final = df[df["is_cancer"] == True].copy()
    
    # Drop the tracking helper column
    df_final.drop(columns=["is_cancer"], inplace=True)

    # ==========================================
    # 4. SAVE CLEAN DATASET
    # ==========================================
    output_filename = "chembl_organometallic_cancer_ic50.csv"
    df_final.to_csv(output_filename, index=False)

    print(f"\n==========================================")
    print(f"SUCCESS!")
    print(f"==========================================")
    print(f"Final Organometallic Cancer Dataset Shape: {df_final.shape}")
    print(f"Saved to: {output_filename}")
    print("\n*Note: Several organometallic compounds will have empty (NaN) SMILES values.")
    print("This occurs because ChEMBL cannot generate standard structural text string rules")
    print("for multi-hapto coordinate bonds (like Ferrocenes or Ruthenium-arenes).")
    
    if len(df_final) > 0:
        print("\nPreview of the first few entries:")
        # Display key tracking columns to inspect data integrity
        print(df_final[['chembl_id', 'chemical_formula', 'ic50_nM', 'pIC50', 'target_name']].head(10))

Connecting to local database at: chembl_36/chembl_36_sqlite/chembl_36.db
Successfully connected.
Executing targeted SQL extraction for metal-bound compounds...
Pulled 70711 initial human records containing target metals in their formulas.
Filtering for cancer-related assays...

SUCCESS!
Final Organometallic Cancer Dataset Shape: (16422, 7)
Saved to: chembl_organometallic_cancer_ic50.csv

*Note: Several organometallic compounds will have empty (NaN) SMILES values.
This occurs because ChEMBL cannot generate standard structural text string rules
for multi-hapto coordinate bonds (like Ferrocenes or Ruthenium-arenes).

Preview of the first few entries:
       chembl_id chemical_formula  ic50_nM  pIC50  \
4   CHEMBL358814     C22H18FN3OS2  47500.0   4.32   
21   CHEMBL11359      H6Cl2N2Pt+2   2300.0   5.64   
22   CHEMBL11359      H6Cl2N2Pt+2   1400.0   5.85   
23   CHEMBL11359      H6Cl2N2Pt+2   3200.0   5.50   
24   CHEMBL11359      H6Cl2N2Pt+2    600.0   6.22   
25   CHEMBL11359      H6Cl

In [12]:
import pandas as pd
import sqlite3
from rdkit import Chem
import re

# ==========================================
# 1. READ YOUR DATA AND RE-CONNECT TO DATABASE
# ==========================================
csv_path = "chembl_organometallic_cancer_ic50.csv"
db_path = "chembl_36/chembl_36_sqlite/chembl_36.db"

df_raw = pd.read_csv(csv_path)
print(f"Loaded {len(df_raw)} rows from your generated CSV.")

conn = sqlite3.connect(db_path)

# ==========================================
# 2. FETCH INCHI STRINGS (CORRECTED COLUMNS)
# ==========================================
# Fixed: Changing 'str.inchi' to ChEMBL's actual column name: 'str.standard_inchi'
print("Fetching Standard InChI representations from local database...")
inchi_query = """
SELECT DISTINCT m.chembl_id, str.standard_inchi, str.standard_inchi_key
FROM molecule_dictionary m
JOIN compound_structures str ON m.molregno = str.molregno;
"""
df_structures = pd.read_sql_query(inchi_query, conn)
conn.close()

# Build index maps for swift lookup execution
struct_map = df_structures.set_index("chembl_id")[["standard_inchi", "standard_inchi_key"]].to_dict(orient="index")

def get_inchi_data(chembl_id):
    return struct_map.get(chembl_id, {"standard_inchi": None, "standard_inchi_key": None})

df_raw["inchi"] = df_raw["chembl_id"].apply(lambda x: get_inchi_data(x)["standard_inchi"])
df_raw["inchi_key"] = df_raw["chembl_id"].apply(lambda x: get_inchi_data(x)["standard_inchi_key"])

# ==========================================
# 3. STRICT RE-FILTER FOR TRUE METALS
# ==========================================
# Eradicates any false-positive organic compounds containing overlapping letters like "pt"
true_metals = {"Pt", "Ru", "Ir", "Au", "Pd", "Os", "Rh", "Re", "Fe", "Co", "Ni"}

def is_true_metal_formula(formula):
    if not isinstance(formula, str):
        return False
    # Tokenizes the alphanumeric fragments to match exact elements
    tokens = set(re.findall(r'[A-Z][a-z]?', formula))
    return not tokens.isdisjoint(true_metals)

df_filtered = df_raw[df_raw["chemical_formula"].apply(is_true_metal_formula)].copy()
print(f"Filtered out false positives. True metal complex rows remaining: {len(df_filtered)}")

# ==========================================
# 4. CONVERT INCHI TO WORKABLE SMILES VIA RDKIT
# ==========================================
def inchi_to_smiles_safe(inchi_str):
    if not isinstance(inchi_str, str):
        return None
    try:
        # sanitize=False forces RDKit to construct the molecule without crashing 
        # over transition metal dative/coordinate valence tracking limits
        mol = Chem.MolFromInchi(inchi_str, sanitize=False, removeHs=True)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol)
    except:
        return None

print("Converting extracted InChI layers to clean structural SMILES strings...")
df_filtered["clean_smiles"] = df_filtered["inchi"].apply(inchi_to_smiles_safe)

# Patch any missing gaps back using the original raw string fallback values
df_filtered["canonical_smiles"] = df_filtered["clean_smiles"].fillna(df_filtered["canonical_smiles"])
df_filtered.drop(columns=["clean_smiles"], inplace=True)

rescued_count = df_filtered["canonical_smiles"].notna().sum()
print(f"Successfully generated/verified molecular graphs for {rescued_count} out of {len(df_filtered)} rows.")

# ==========================================
# 5. SAVE COMPLETED DATASET
# ==========================================
final_output = "chembl_organometallic_cancer_final_structures.csv"
df_filtered.to_csv(final_output, index=False)

print(f"\n==========================================")
print(f"CLEANING COMPLETE!")
print(f"==========================================")
print(f"Final usable dataset saved to: {final_output}")
print("\nPreview of cleaned metal-complex entries:")
print(df_filtered[['chembl_id', 'chemical_formula', 'canonical_smiles', 'ic50_nM', 'pIC50']].head(5))

Loaded 16422 rows from your generated CSV.
Fetching Standard InChI representations from local database...
Filtered out false positives. True metal complex rows remaining: 1816
Converting extracted InChI layers to clean structural SMILES strings...
Successfully generated/verified molecular graphs for 0 out of 1816 rows.

CLEANING COMPLETE!
Final usable dataset saved to: chembl_organometallic_cancer_final_structures.csv

Preview of cleaned metal-complex entries:
     chembl_id chemical_formula canonical_smiles  ic50_nM  pIC50
1  CHEMBL11359      H6Cl2N2Pt+2              NaN   2300.0   5.64
2  CHEMBL11359      H6Cl2N2Pt+2              NaN   1400.0   5.85
3  CHEMBL11359      H6Cl2N2Pt+2              NaN   3200.0   5.50
4  CHEMBL11359      H6Cl2N2Pt+2              NaN    600.0   6.22
5  CHEMBL11359      H6Cl2N2Pt+2              NaN   1900.0   5.72


In [14]:
import pandas as pd
import re

# ==========================================
# 1. LOAD DATA
# ==========================================
csv_path = "chembl_organometallic_cancer_final_structures.csv"
df = pd.read_csv(csv_path)

print(f"Processing {len(df)} true organometallic records...")

# Drop the unworkable smiles column
if 'canonical_smiles' in df.columns:
    df.drop(columns=['canonical_smiles'], inplace=True)

# ==========================================
# 2. FEATURE ENGINEERING FROM CHEMICAL FORMULA
# ==========================================
# Dictionary of standard atomic weights for molecular weight calculation
atomic_weights = {
    'H': 1.008, 'He': 4.0026, 'Li': 6.94, 'Be': 9.0122, 'B': 10.81, 'C': 12.011,
    'N': 14.007, 'O': 15.999, 'F': 18.998, 'Ne': 20.180, 'Na': 22.990, 'Mg': 24.305,
    'Al': 26.982, 'Si': 28.085, 'P': 30.974, 'S': 32.06, 'Cl': 35.45, 'Ar': 39.948,
    'K': 39.098, 'Ca': 40.078, 'Sc': 44.956, 'Ti': 47.867, 'V': 50.942, 'Cr': 51.996,
    'Mn': 54.938, 'Fe': 55.845, 'Co': 58.933, 'Ni': 58.693, 'Cu': 63.546, 'Zn': 65.38,
    'Ga': 69.723, 'Ge': 72.63, 'As': 74.922, 'Se': 78.971, 'Br': 79.904, 'Kr': 83.798,
    'Y': 88.906, 'Zr': 91.224, 'Nb': 92.906, 'Mo': 95.95, 'Tc': 97, 'Ru': 101.07,
    'Rh': 102.91, 'Pd': 106.42, 'Ag': 107.87, 'Cd': 112.41, 'In': 114.82, 'Sn': 118.71,
    'Sb': 121.76, 'Te': 127.60, 'I': 126.90, 'Xe': 131.29, 'Cs': 132.91, 'Ba': 137.33,
    'La': 138.91, 'Ce': 140.12, 'Pr': 140.91, 'Nd': 144.24, 'Pm': 145, 'Sm': 150.36,
    'Eu': 151.96, 'Gd': 157.25, 'Tb': 158.93, 'Dy': 162.50, 'Ho': 164.93, 'Er': 167.26,
    'Tm': 168.93, 'Yb': 173.05, 'Lu': 174.97, 'Hf': 178.49, 'Ta': 180.95, 'W': 183.84,
    'Re': 186.21, 'Os': 190.23, 'Ir': 192.22, 'Pt': 195.08, 'Au': 196.97, 'Hg': 200.59,
    'Tl': 204.38, 'Pb': 207.2, 'Bi': 208.98
}

def parse_formula_features(formula):
    if not isinstance(formula, str):
        return 0.0, 0, 0 # MW, Atom Count, Charge
    
    # 1. Handle charges (e.g., +2, -1)
    charge = 0
    charge_match = re.search(r'([\+\-])(\d*)', formula)
    if charge_match:
        sign, val = charge_match.groups()
        val = int(val) if val else 1
        charge = val if sign == '+' else -val
        # Strip charge details from formula text processing
        formula = re.sub(r'[\+\-]\d*', '', formula)
        
    # 2. Parse elements and numbers
    matches = re.findall(r'([A-Z][a-z]*)(\d*)', formula)
    
    mw = 0.0
    total_atoms = 0
    
    for element, count in matches:
        count = int(count) if count else 1
        total_atoms += count
        mw += atomic_weights.get(element, 0.0) * count
        
    return mw, total_atoms, charge

print("Calculating molecular weights and atom counts from formulas...")
formula_data = df["chemical_formula"].apply(parse_formula_features)
df["computed_molecular_weight"] = [x[0] for x in formula_data]
df["total_atom_count"] = [x[1] for x in formula_data]
df["net_charge"] = [x[2] for x in formula_data]

# ==========================================
# 3. IDENTIFY AND ONE-HOT ENCODE TARGET METALS
# ==========================================
target_metals = ["Pt", "Ru", "Ir", "Au", "Pd", "Os", "Rh", "Re", "Fe", "Co", "Ni"]

def get_metal_type(formula):
    if not isinstance(formula, str):
        return "Unknown"
    for metal in target_metals:
        if metal in formula:
            return metal
    return "Other"

df["metal_type"] = df["chemical_formula"].apply(get_metal_type)

# Perform one-hot encoding for the metal types
df = pd.get_dummies(df, columns=["metal_type"], prefix="metal", dtype=int)

# ==========================================
# 4. HANDLE MISSING pIC50 VALUES
# ==========================================
# If pIC50 is missing, compute it mathematically from ic50_nM
# pIC50 = -log10(IC50_Molar) -> ic50_nM * 1e-9
import numpy as np
df["pIC50"] = df["pIC50"].fillna(-np.log10(df["ic50_nM"] * 1e-9))

# Clean out any extreme infinite outliers or negative numbers
df = df[(df["pIC50"] > 0) & (df["pIC50"] < 15)]

# ==========================================
# 5. SAVE MACHINE LEARNING READY DATASET
# ==========================================
ml_output = "organometallic_ml_ready_matrix.csv"
df.to_csv(ml_output, index=False)

print(f"\n==========================================")
print(f"ML READY MATRIX GENERATED SUCCESSFULLY!")
print(f"==========================================")
print(f"Final Data Matrix Shape: {df.shape}")
print(f"Saved to: {ml_output}")
print("\nAvailable Features for Model Training:")
feature_cols = ['computed_molecular_weight', 'total_atom_count', 'net_charge'] + [c for c in df.columns if 'metal_' in c]
print(feature_cols)
print("\nPreview:")
print(df[['chembl_id', 'chemical_formula', 'ic50_nM', 'pIC50', 'computed_molecular_weight', 'net_charge']].head(5))

Processing 1816 true organometallic records...
Calculating molecular weights and atom counts from formulas...

ML READY MATRIX GENERATED SUCCESSFULLY!
Final Data Matrix Shape: (1816, 17)
Saved to: organometallic_ml_ready_matrix.csv

Available Features for Model Training:
['computed_molecular_weight', 'total_atom_count', 'net_charge', 'metal_Au', 'metal_Fe', 'metal_Os', 'metal_Pd', 'metal_Pt', 'metal_Ru']

Preview:
     chembl_id chemical_formula  ic50_nM  pIC50  computed_molecular_weight  \
0  CHEMBL11359      H6Cl2N2Pt+2   2300.0   5.64                    300.042   
1  CHEMBL11359      H6Cl2N2Pt+2   1400.0   5.85                    300.042   
2  CHEMBL11359      H6Cl2N2Pt+2   3200.0   5.50                    300.042   
3  CHEMBL11359      H6Cl2N2Pt+2    600.0   6.22                    300.042   
4  CHEMBL11359      H6Cl2N2Pt+2   1900.0   5.72                    300.042   

   net_charge  
0           2  
1           2  
2           2  
3           2  
4           2  
